In [1]:
import numpy as np
import os
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

# [단계 1] 동작 정의 및 데이터 불러오기
# 수집할 때 사용했던 순서와 똑같아야 합니다.
actions = ['scroll_down', 'scroll_up', 'next_ep', 'prev_ep', 'menu']

data = []
for action in actions:
    # dataset 폴더 내의 해당 동작 파일들을 모두 찾아서 합칩니다.
    # 여러 번 수집했을 경우를 대비해 파일 이름이 seq_동작명으로 시작하는 것들을 다 가져옵니다.
    file_list = [f for f in os.listdir('dataset') if f.startswith(f'seq_{action}_')]
    for file in file_list:
        file_path = os.path.join('dataset', file)
        temp_data = np.load(file_path)
        data.extend(temp_data)

data = np.array(data)

# 데이터 구조 확인 (총 데이터 개수, 30프레임, 100개 특징점)
print(f"전체 데이터 모양: {data.shape}")


전체 데이터 모양: (4320, 30, 100)


In [2]:
# [단계 2] 특징 데이터(X)와 정답 데이터(y) 분리
# 데이터의 마지막 컬럼은 우리가 저장할 때 넣었던 동작의 인덱스(0~4)입니다.
X = data[:, :, :-1].astype(np.float32) # 마지막 한 칸을 제외한 모든 좌표값
y = data[:, 0, -1].astype(np.int32)    # 각 시퀀스의 첫 번째 프레임에 기록된 정답 인덱스

# 정답 데이터를 원-핫 인코딩 (예: 2 -> [0, 0, 1, 0, 0])
y_onehot = to_categorical(y, num_classes=len(actions))

# 학습 데이터와 검증 데이터로 나눕니다 (80% 학습, 20% 테스트)
X_train, X_test, y_train, y_test = train_test_split(X, y_onehot, test_size=0.2, random_state=42)

print(f"학습 데이터: {X_train.shape}, 테스트 데이터: {X_test.shape}")

학습 데이터: (3456, 30, 99), 테스트 데이터: (864, 30, 99)


In [3]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau

# [단계 3] 모델 설계
model = Sequential([
    # 첫 번째 레이어에서 ReLU를 뺐던 경험을 살려, 기본 활성화 함수(tanh)를 사용합니다.
    LSTM(64, activation='tanh', input_shape=X_train.shape[1:]),
    Dropout(0.2), # 과적합을 방지하기 위해 공부할 때 20%는 무작위로 빼먹게 합니다.
    Dense(32, activation='relu'),
    Dense(len(actions), activation='softmax') # 최종 결과는 5개 동작 중 하나로 출력
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

# [단계 4] 학습 시작
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=100, # 100번 반복 학습
    batch_size=32,
    callbacks=[
        ModelCheckpoint('models/webtoon_model.h5', monitor='val_loss', save_best_only=True, mode='min', verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, verbose=1, mode='min')
    ]
)



Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm (LSTM)                 (None, 64)                41984     
                                                                 
 dropout (Dropout)           (None, 64)                0         
                                                                 
 dense (Dense)               (None, 32)                2080      
                                                                 
 dense_1 (Dense)             (None, 5)                 165       
                                                                 
Total params: 44229 (172.77 KB)
Trainable params: 44229 (172.77 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________
Epoch 1/100


105/108 [============================>.] - ETA: 0s - loss: 0.5224 - accuracy: 0.8860
Epoch 1: val_loss improved from inf to 0.04214, s

c:\Users\10neu\anaconda3\envs\py311\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


108/108 [==============================] - ETA: 0s - loss: 0.0315 - accuracy: 0.9994
Epoch 2: val_loss improved from 0.04214 to 0.00667, saving model to models\webtoon_model.h5
108/108 [==============================] - 1s 8ms/step - loss: 0.0315 - accuracy: 0.9994 - val_loss: 0.0067 - val_accuracy: 1.0000 - lr: 0.0010
Epoch 3/100
102/108 [===========================>..] - ETA: 0s - loss: 0.0093 - accuracy: 1.0000
Epoch 3: val_loss improved from 0.00667 to 0.00226, saving model to models\webtoon_model.h5
108/108 [==============================] - 1s 7ms/step - loss: 0.0090 - accuracy: 1.0000 - val_loss: 0.0023 - val_accuracy: 1.0000 - lr: 0.0010
Epoch 4/100
103/108 [===========================>..] - ETA: 0s - loss: 0.0054 - accuracy: 1.0000
Epoch 4: val_loss improved from 0.00226 to 0.00115, saving model to models\webtoon_model.h5
108/108 [==============================] - 1s 7ms/step - loss: 0.0052 - accuracy: 1.0000 - val_loss: 0.0011 - val_accuracy: 1.0000 - lr: 0.0010
Epoch 5/100
1